#Imports 

In [0]:
# init
from pyspark.sql import functions as F
from pyspark.sql.functions import upper, lower, col, when, current_date
from pyspark.sql.types import IntegerType, StringType, DataType


# Read file from Bronze Layer

In [0]:
df = spark.table("workspace.bronze_schema.erp_cust_az12")

In [0]:
df.display()

# Explore Data to find inconsistencies

In [0]:
# Describe the df stats
df.describe().display()
"""
    As deduced from the describe the GEN column has min value which seems to be an empty string. 
    This is to be explored later when working with the GEN column
"""

In [0]:
df.printSchema()

"""
    Column names are not proper. Column name remapping to be done after transformation.
"""

## Checking for white spaces

In [0]:
for field in df.schema:
    column_name = field.name
    column_data_type = field.dataType

    if isinstance(column_data_type, StringType):
        white_space_df = df.select("*").where(F.trim(col(column_name)) != col(column_name))
        white_space_count = white_space_df.count()

        if white_space_count:
            print(f"White space of count {white_space_count} found in Column {column_name}")
            white_space_df.display()

"""
    Some white space found in the GEN column.. Needs to be fixed
"""

## Checking for null or missing values

In [0]:
for column in df.columns:
    null_df = df.select("*").where(df[column].isNull())
    null_df_count = null_df.count()

    if null_df_count:
        print(f"Null values of count {null_df_count} found in Column {column}")
        null_df.display()
    


"""
    Null values found in the GEN column
"""

## Check if CID matches customer_key in the crm_cust_info table

In [0]:
# 
df.createOrReplaceTempView("erp_cust_az12_table")

query = """
    SELECT *
    FROM erp_cust_az12_table
    WHERE CID NOT IN (
            SELECT customer_key 
            FROM workspace.silver_schema.crm_cust_info
        )
"""

unmatched_CID = spark.sql(query)
print(f"{unmatched_CID.count()} CID unmatched")
unmatched_CID.display()

"""
    Results show that some of the CID does not match the customer_key on the crm_cust_info.
    Removing the NOT from the WHERE clause shows that only CID LIKE AW% have matches.
    So we see that we have some CID LIKE NAS% and other LIKE AW%

"""

## Check how many CID LIKE AW% and NAS%

In [0]:
AW_df = df.select("*").where(col("CID").like("AW%"))
print(f"{AW_df.count()} rows have CID LIKE AW%")
AW_df.display()


NAS_df = df.select("*").where(col("CID").like("NAS%"))
print(f"{NAS_df.count()} rows have CID LIKE NAS%")
NAS_df.display()

"""
    From the results of cell 14 and 16, it can be deduces that all NAS% LIKE CID have no matches as we have 11042 rows from both queries
"""


## Check BDATE to make sure not on is born after today curr date

In [0]:
df.select("*", current_date().alias("curr_date")).where(col("BDATE") > current_date()).display()

"""
    We can see that some BDATE are wrong as they are in the future (Greater than the curr_date)
"""

## Checking for distinct values in the GEN column

In [0]:
df.select((col("GEN"))).distinct().display()

"""
    There are multiple variations of Male and Female, Some as a result of white space that needs to be trimmed. We can also find missing values and blank spaces.
"""

# Data Notes
- GEN column has empty strings, null values,  white spaces, and there are multiple variations of Male and Female,
- Columns are not properly named. Needs renaming
- Some CID does not match customer_key from crm_cust_info table as in accordance with the Entity Relationship Diagram (ERD). AW% Match but NAS% does not match. Might remove NAS to see if they match.
- Some BDATE are wrong as they are in the future (Greater than the curr_date)

# Data Transformations

# Write transformed data to the Silver Layer